In [1]:
import numpy as np

N =  8

In [2]:
from Age_BMI_loading import (
    age_matrix_vec,
    bmi_matrix_ls,
    age_dict,
    bmi_samples_dict
)

In [3]:
from impute_diabetes import (
    schedule_pre_diabetes_probability,
    schedule_diabetes_probability
)

In [4]:
# get 1990 

diabetes_status_dict = {}
diabetes_mat_list = []
for group in range(N):
        i = group
        diabetes_status_dict[group] = {}
        diabetes_status = np.zeros((age_matrix_vec[i].shape[0], age_matrix_vec[i].shape[1]))  
        for age in age_dict[group]:
                bmi_samples = bmi_samples_dict[group][age]  # shape: (n_samples, n_ages)
                n_samples, n_ages = bmi_samples.shape
                # Create an age matrix with shape (n_samples, n_ages)
                n_samples = age_dict[group][age]
                age_mat = np.tile(np.arange(18, age + 1), (n_samples, 1))
                # idx = group

                # Compute prediabetes probability matrix
                prediabetes_prob_matrix = schedule_pre_diabetes_probability(age_mat,i,bmi_samples)
                rand_matrix = np.random.rand(*bmi_samples.shape)
                prediabetes_matrix = (rand_matrix < prediabetes_prob_matrix) * 0.5
                prediabetes_matrix = np.maximum.accumulate(prediabetes_matrix, axis=1)

                # Compute diabetes probability matrix
                diabetes_prob_matrix = schedule_diabetes_probability(age_mat,i,bmi_samples)
                rand_matrix = np.random.rand(*bmi_samples.shape)
                diabetes_matrix = np.where(rand_matrix < diabetes_prob_matrix, 1, 0.5)

                # Combine: take max along each row (for each sample)
                combined_matrix = np.where(prediabetes_matrix == 0.5, diabetes_matrix, prediabetes_matrix)
                combined_matrix = np.maximum.accumulate(combined_matrix, axis=1)

                diabetes_status_dict[group][age] = np.max(combined_matrix, axis=1)
        


In [5]:
diabetes_status_dict[0][18].shape

(922,)

In [6]:
positions = np.where(age_matrix_vec[i][:, 0] == age)[0]
diabetes_status[positions, 0].shape

(2,)

In [7]:
diabetes_mat_list = []
prediabetes_mat_list = []
for i in range(8):
        diabetes_status = np.zeros((age_matrix_vec[i].shape[0], age_matrix_vec[i].shape[1]))  
        for age, status_vec in diabetes_status_dict[i].items():
            # Find positions where the first column of age_matrix_vec[i] equals 'age'
            positions = np.where(age_matrix_vec[i][:, 0] == age)[0]
            # Assign the diabetes status vector for this age group
            diabetes_status[positions, 0] = status_vec

        n_cols = age_matrix_vec[i].shape[1]
        diabetes_status_before = np.repeat(diabetes_status[:, 0][:, np.newaxis], n_cols, axis=1)
        # Process diabetes status
        bmi_mat = bmi_matrix_ls[i]
        age_mat = age_matrix_vec[i]

        prediabetes_prob_matrix = schedule_pre_diabetes_probability(age_mat, i, bmi_mat)
        rand_matrix = np.random.rand(*age_mat.shape)
        prediabetes_matrix = (rand_matrix < prediabetes_prob_matrix) * 0.5
        
        prediabetes_matrix = np.where(diabetes_status_before == 0, prediabetes_matrix, diabetes_status_before)
        prediabetes_matrix = np.maximum.accumulate(prediabetes_matrix, axis=1)
        prediabetes_mat_list.append(prediabetes_matrix)
        
        diabetes_prob_matrix = schedule_diabetes_probability(age_mat, i, bmi_mat)
        rand_matrix = np.random.rand(*age_mat.shape)
        diabetes_matrix = np.where(rand_matrix < diabetes_prob_matrix, 1, 0.5)
        
        combined_matrix = np.where(prediabetes_matrix == 0.5, diabetes_matrix, prediabetes_matrix)

        diabetes_status[:, 1:] = combined_matrix[:, 1:]
        diabetes_status = np.maximum.accumulate(diabetes_status, axis=1)
        diabetes_mat_list.append(diabetes_status)
    

In [8]:
from prevalence import (
    get_prevalence,
    get_pre_prevalence
)

In [9]:
diabetes_final_result = get_prevalence(age_matrix_vec,diabetes_mat_list,-2)

In [10]:
diabetes_final_result

array([0.08523041, 0.05894623, 0.13648172, 0.12008166, 0.19522016,
       0.14906219, 0.00059199, 0.01198441, 0.04378904, 0.11296195,
       0.212145  , 0.29637349])

In [11]:
targeting_diabeteslist = np.array([ 8.2, 5.8, 13.5, 12.4, 19.1, 14.4, 0.2 , 1.9, 5.0, 10.8, 21.8, 24.2])

In [12]:
import os
# Create the directory if it doesn't exist
save_dir = "../data/diabetes_matrix"
os.makedirs(save_dir, exist_ok=True)

# Save each matrix in diabetes_mat_list
for i, diabetes_mat in enumerate(diabetes_mat_list):
    save_path = os.path.join(save_dir, f"diabetes_mat_{i}.npy")
    np.save(save_path, diabetes_mat)
    print(f"Saved: {save_path}")

Saved: ../data/diabetes_matrix/diabetes_mat_0.npy
Saved: ../data/diabetes_matrix/diabetes_mat_1.npy
Saved: ../data/diabetes_matrix/diabetes_mat_2.npy
Saved: ../data/diabetes_matrix/diabetes_mat_3.npy
Saved: ../data/diabetes_matrix/diabetes_mat_4.npy
Saved: ../data/diabetes_matrix/diabetes_mat_5.npy
Saved: ../data/diabetes_matrix/diabetes_mat_6.npy
Saved: ../data/diabetes_matrix/diabetes_mat_7.npy
